<a href="https://colab.research.google.com/github/yosepdoni/Advanced-Artificial-Intelligence/blob/main/RawNet3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
# ========================================
# DETEKSI SUARA SINTETIS - VERSION 2 (ERROR HANDLING)
# ========================================

print("📦 Installing libraries...")
!pip install -q librosa soundfile gtts scikit-learn joblib

import librosa
import numpy as np
import pandas as pd
from gtts import gTTS
import os
import glob
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries installed!\n")

# ========================================
# GENERATE SYNTHETIC
# ========================================
print("🔄 Generating synthetic voices...")
os.makedirs('synthetic', exist_ok=True)
os.makedirs('real', exist_ok=True)

texts = [
    "Halo nama saya Budi berusia dua puluh lima tahun",
    "Selamat pagi saya mahasiswa dari Jakarta",
    "Perkenalkan saya bekerja sebagai guru",
    "Saya suka belajar tentang kecerdasan buatan",
    "Teknologi suara sangat menarik untuk dipelajari",
    "Indonesia memiliki banyak bahasa daerah",
    "Saya senang mendengarkan musik tradisional",
    "Penelitian ini mendeteksi suara sintetis",
    "Terima kasih sudah mendengarkan",
    "Sampai jumpa di lain waktu",
    "Cuaca hari ini sangat cerah sekali",
    "Saya sedang belajar machine learning",
    "Universitas saya berada di kota besar",
    "Hobi saya adalah membaca buku",
    "Semoga penelitian ini bermanfaat"
]

for i, text in enumerate(texts):
    tts = gTTS(text=text, lang='id', slow=False)
    tts.save(f'synthetic/synth_{i}.wav')

print(f"✅ Generated {len(texts)} synthetic voices\n")

# ========================================
# UPLOAD REAL
# ========================================
from google.colab import files

print("=" * 70)
print("📤 UPLOAD MINIMAL 10 REKAMAN SUARA ASLI MANUSIA")
print("=" * 70)
print("💡 TIPS:")
print("   - Minimal 10 file (lebih banyak lebih baik)")
print("   - Format: .wav atau .mp3")
print("   - Durasi: 3-10 detik")
print("   - Bisa dari: HP, Common Voice, YouTube\n")

uploaded = files.upload()

if len(uploaded) == 0:
    print("\n❌ ERROR: Tidak ada file yang di-upload!")
    print("Silakan upload minimal 10 rekaman suara asli")
    raise ValueError("No files uploaded")

for filename in uploaded.keys():
    with open(f'real/{filename}', 'wb') as f:
        f.write(uploaded[filename])

print(f"\n✅ Uploaded {len(uploaded)} real voice samples")

if len(uploaded) < 5:
    print(f"\n⚠️  WARNING: Hanya {len(uploaded)} file!")
    print("   Untuk hasil terbaik, upload minimal 10 file")
    print("   Lanjut dengan data yang ada...\n")

# ========================================
# EXTRACT FEATURES
# ========================================
print("🔄 Extracting features...")

def extract_features(audio_path):
    try:
        y, sr = librosa.load(audio_path, sr=16000, duration=5)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        mfcc_mean = np.mean(mfcc, axis=1)

        spectral_centroid = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
        spectral_contrast = np.mean(librosa.feature.spectral_contrast(y=y, sr=sr))
        zero_crossing = np.mean(librosa.feature.zero_crossing_rate(y))

        pitches = librosa.piptrack(y=y, sr=sr)[0]
        pitch_mean = np.mean(pitches[pitches > 0]) if np.any(pitches > 0) else 0

        rms = np.mean(librosa.feature.rms(y=y))
        D = librosa.stft(y)
        phase_std = np.std(np.angle(D))

        features = np.concatenate([
            mfcc_mean,
            [spectral_centroid, spectral_contrast, zero_crossing,
             pitch_mean, rms, phase_std]
        ])
        return features
    except Exception as e:
        print(f"   ⚠️  Error di {os.path.basename(audio_path)}: {e}")
        return None

X, y = [], []

for audio in glob.glob('synthetic/*.wav'):
    feat = extract_features(audio)
    if feat is not None:
        X.append(feat)
        y.append(1)

for audio in glob.glob('real/*.wav') + glob.glob('real/*.mp3'):
    feat = extract_features(audio)
    if feat is not None:
        X.append(feat)
        y.append(0)

X = np.array(X)
y = np.array(y)

n_real = np.sum(y == 0)
n_synthetic = np.sum(y == 1)

print(f"✅ Features extracted:")
print(f"   - Real: {n_real} samples")
print(f"   - Synthetic: {n_synthetic} samples")
print(f"   - Total: {len(X)} samples\n")

if n_real == 0:
    print("❌ ERROR: Tidak ada real samples yang berhasil di-extract!")
    print("Kemungkinan file corrupt atau format tidak didukung")
    raise ValueError("No valid real samples")

# ========================================
# TRAIN MODEL (FIXED)
# ========================================
print("🤖 Training model...")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Adaptive training based on data size
if len(X) < 10 or n_real < 3 or n_synthetic < 3:
    print("⚠️  Data terlalu sedikit untuk train-test split")
    print("   Menggunakan cross-validation\n")

    model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
    model.fit(X_scaled, y)

    cv_scores = cross_val_score(model, X_scaled, y, cv=min(3, len(X)//2))
    accuracy = cv_scores.mean() * 100

    # Untuk evaluasi
    y_pred = model.predict(X_scaled)
    y_test = y

    print(f"✅ Model trained!")
    print(f"   CV Accuracy: {accuracy:.2f}%\n")

else:
    # Normal train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.3, random_state=42
    )

    model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred) * 100

    print(f"✅ Model trained!")
    print(f"   Test Accuracy: {accuracy:.2f}%\n")

# ========================================
# EVALUATE
# ========================================
print("=" * 70)
print("📊 HASIL EVALUASI")
print("=" * 70)
print(f"\n✅ AKURASI: {accuracy:.2f}%\n")

if len(np.unique(y_test)) == 2:
    print("📋 Classification Report:")
    print(classification_report(y_test, y_pred,
                              target_names=['Real', 'Synthetic'],
                              digits=2, zero_division=0))

    print("\n📊 Confusion Matrix:")
    cm = confusion_matrix(y_test, y_pred)
    print(cm)
else:
    print("⚠️  Evaluasi terbatas karena data sedikit")

# ========================================
# SAVE & TEST
# ========================================
joblib.dump(model, 'model.pkl')
joblib.dump(scaler, 'scaler.pkl')

def predict_new(audio_path):
    feat = extract_features(audio_path)
    if feat is None:
        return "ERROR"
    feat_scaled = scaler.transform(feat.reshape(1, -1))
    pred = model.predict(feat_scaled)[0]
    prob = model.predict_proba(feat_scaled)[0][pred]
    label = 'SYNTHETIC' if pred == 1 else 'REAL'
    return f"{label} ({prob*100:.1f}%)"

print("\n✅ Model tersimpan!")
print(f"   Siap digunakan dengan fungsi predict_new()\n")

# Test
print("🔍 Testing samples:")
for audio in (glob.glob('synthetic/*.wav')[:3] + glob.glob('real/*')[:3]):
    print(f"   {os.path.basename(audio):30s} → {predict_new(audio)}")

print("\n🎉 SELESAI! Akurasi: {:.1f}%".format(accuracy))

📦 Installing libraries...
✅ Libraries installed!

🔄 Generating synthetic voices...
✅ Generated 15 synthetic voices

📤 UPLOAD MINIMAL 10 REKAMAN SUARA ASLI MANUSIA
💡 TIPS:
   - Minimal 10 file (lebih banyak lebih baik)
   - Format: .wav atau .mp3
   - Durasi: 3-10 detik
   - Bisa dari: HP, Common Voice, YouTube



Saving Recording (3) - Copy - Copy.wav to Recording (3) - Copy - Copy.wav
Saving Recording (3) - Copy.m4a to Recording (3) - Copy.m4a
Saving Recording (3) - Copy.wav to Recording (3) - Copy.wav
Saving Recording (3).m4a to Recording (3) (1).m4a
Saving Recording (3).wav to Recording (3) (1).wav
Saving Recording (4) - Copy - Copy.wav to Recording (4) - Copy - Copy.wav
Saving Recording (4) - Copy (2) - Copy.wav to Recording (4) - Copy (2) - Copy.wav
Saving Recording (4) - Copy (2).wav to Recording (4) - Copy (2).wav
Saving Recording (4) - Copy.wav to Recording (4) - Copy.wav
Saving Recording (4).wav to Recording (4) (4).wav

✅ Uploaded 10 real voice samples
🔄 Extracting features...
✅ Features extracted:
   - Real: 11 samples
   - Synthetic: 15 samples
   - Total: 26 samples

🤖 Training model...
✅ Model trained!
   Test Accuracy: 100.00%

📊 HASIL EVALUASI

✅ AKURASI: 100.00%

📋 Classification Report:
              precision    recall  f1-score   support

        Real       1.00      1.00   